# 09 — From Scores to Attention Patterns

**Description:** Convert query–key scores into attention weights using scaling, causal masking, and row-wise softmax.
**Level:** Beginner
**Tags:** Language Models, Attention, Softmax, Causal Masking, Visualization

Notebook 08 produced the raw score matrix $QK^T$. This notebook introduces one pipeline:

$$QK^T ightarrow 	ext{scale} ightarrow 	ext{mask} ightarrow 	ext{softmax} ightarrow A$$

By the end, you will be able to implement scaled dot-product attention weights, explain causal masking, and verify the invariants of an attention matrix. We still will not introduce values; that is Notebook 10.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

np.set_printoptions(precision=3, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")

## 1. Recreate the queries and keys

We reuse the deterministic example from Notebook 08 so the only new ideas are the transformations applied to its scores.

In [ ]:
tokens = ["the", "robot", "fixed", "it"]
X = np.array([[1.0, 0.0, 0.2], [0.2, 1.0, 0.6], [0.1, 0.7, 1.0], [0.8, 0.2, 0.4]])
W_Q = np.array([[1.0, 0.0], [0.0, 0.8], [0.5, 0.5]])
W_K = np.array([[0.6, 0.2], [0.1, 1.0], [0.8, 0.3]])
Q, K = X @ W_Q, X @ W_K
raw_scores = Q @ K.T
d_k = Q.shape[-1]

print("raw scores shape:", raw_scores.shape)
print(raw_scores)

## 2. Why divide by $\sqrt{d_k}$?

A dot product adds $d_k$ products. As $d_k$ grows, its typical magnitude tends to grow too. Large score gaps can make softmax extremely sharp. Scaled dot-product attention uses:

$$S = rac{QK^T}{\sqrt{d_k}}$$

The scale does not change which score is largest; it moderates the gaps.

In [ ]:
scaled_scores = raw_scores / np.sqrt(d_k)
print("d_k:       ", d_k)
print("scale:     ", np.sqrt(d_k))
print("raw row:   ", raw_scores[-1])
print("scaled row:", scaled_scores[-1])
assert np.argmax(raw_scores[-1]) == np.argmax(scaled_scores[-1])

### A small simulation

Independent random coordinates make the stabilizing effect visible. The standard deviation of unscaled dot products grows with width, while scaling keeps it roughly steady.

In [ ]:
rng = np.random.default_rng(9)
widths = np.array([2, 8, 32, 128, 512])
raw_stds, scaled_stds = [], []
for width in widths:
    q = rng.normal(size=(5000, width))
    k = rng.normal(size=(5000, width))
    dots = np.sum(q * k, axis=1)
    raw_stds.append(dots.std())
    scaled_stds.append((dots / np.sqrt(width)).std())

plt.plot(widths, raw_stds, "o-", label="raw dot products")
plt.plot(widths, scaled_stds, "o-", label=r"divided by $\sqrt{d_k}$")
plt.xscale("log", base=2)
plt.xlabel("key/query width $d_k$")
plt.ylabel("standard deviation")
plt.title("Scaling controls score magnitude")
plt.legend()
plt.show()

## 3. Softmax creates one distribution per query

Apply softmax across the **key/source columns** (`axis=-1`). Every row then becomes non-negative and sums to one.

In [ ]:
def softmax(values, axis=-1):
    shifted = values - np.max(values, axis=axis, keepdims=True)
    exponentials = np.exp(shifted)
    return exponentials / exponentials.sum(axis=axis, keepdims=True)

unmasked_attention = softmax(scaled_scores, axis=-1)
print(unmasked_attention)
print("row sums:", unmasked_attention.sum(axis=-1))
assert np.allclose(unmasked_attention.sum(axis=-1), 1.0)
assert np.all(unmasked_attention >= 0)

This is **bidirectional** attention: any position may use any other position, including positions to its right. That is useful in some models, but it would let a next-token model peek at future tokens during training.

## 4. A causal mask blocks the future

At target position $i$, a causal language model may use sources $j\le i$, but not $j>i$. The forbidden cells form the upper triangle above the diagonal.

In [ ]:
sequence_length = len(tokens)
future_is_forbidden = np.triu(
    np.ones((sequence_length, sequence_length), dtype=bool), k=1
)
print(future_is_forbidden.astype(int))

We replace forbidden scores with $-\infty$ **before** softmax. Since $e^{-\infty}=0$, forbidden positions receive exactly zero probability.

In [ ]:
masked_scores = np.where(future_is_forbidden, -np.inf, scaled_scores)
causal_attention = softmax(masked_scores, axis=-1)

print("masked scores:\n", masked_scores)
print("causal attention:\n", causal_attention)
assert np.allclose(causal_attention[future_is_forbidden], 0.0)
assert np.allclose(causal_attention.sum(axis=-1), 1.0)

### Why mask before softmax?

Zeroing probabilities after softmax makes each row sum to less than one unless it is renormalized. Masking scores first lets softmax normalize only over allowed positions.

In [ ]:
wrong = unmasked_attention.copy()
wrong[future_is_forbidden] = 0.0
print("row sums after zeroing too late:", wrong.sum(axis=-1))
print("correct causal row sums:        ", causal_attention.sum(axis=-1))

## 5. Compare masked and unmasked patterns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
for ax, matrix, title in zip(axes, [unmasked_attention, causal_attention], ["Unmasked", "Causal"]):
    image = ax.imshow(matrix, cmap="Blues", vmin=0, vmax=max(unmasked_attention.max(), causal_attention.max()))
    ax.set_xticks(range(len(tokens)), tokens)
    ax.set_yticks(range(len(tokens)), tokens)
    ax.set(xlabel="Key / source", ylabel="Query / target", title=title)
    for i in range(len(tokens)):
        for j in range(len(tokens)):
            ax.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center")
fig.colorbar(image, ax=axes, label="attention weight")
plt.show()

The first causal row must put all weight on `the`: it has no earlier context. Later rows may distribute weight across a larger prefix. The diagonal remains available because a token usually needs access to its own current representation.

## 6. Package the pipeline

This function produces attention **weights**, not updated embeddings. Notice the precise order of operations.

In [ ]:
def attention_weights(Q, K, causal=False):
    if Q.shape[-1] != K.shape[-1]:
        raise ValueError("queries and keys must have the same width")
    scores = Q @ K.T / np.sqrt(Q.shape[-1])
    if causal:
        mask = np.triu(np.ones(scores.shape, dtype=bool), k=1)
        scores = np.where(mask, -np.inf, scores)
    return softmax(scores, axis=-1)

A = attention_weights(Q, K, causal=True)
print(A)
assert np.allclose(A, causal_attention)

## 7. Common axis mistakes

Row-wise softmax answers “where should this query gather from?” Column-wise softmax answers a different question and does not give one distribution per query.

In [ ]:
column_softmax = softmax(scaled_scores, axis=0)
print("column-softmax row sums:   ", column_softmax.sum(axis=1))
print("column-softmax column sums:", column_softmax.sum(axis=0))

## 8. Challenges

1. Confirm that multiplying all scores by 10 makes each unmasked row sharper.
2. Create a mask that lets each token attend only to itself and the immediately previous token.
3. Explain why the causal mask depends on positions, not token identities.
4. Implement the pipeline in PyTorch with `torch.softmax` and compare results.

## Takeaways

- Divide $QK^T$ by $\sqrt{d_k}$ to keep score magnitudes controlled.
- Apply a causal mask before softmax by placing $-\infty$ in future positions.
- Apply softmax across source columns so every query row is a distribution.
- An attention matrix is non-negative, each row sums to one, and causal future entries are zero.
- The matrix now says **where to read**. Notebook 10 adds values—the information that gets read.